# Rebuild the Full Detector Bank Using the Locked ACC+ HSNNET Model

This notebook rebuilds the five-algorithm detector bank using the same locked **`hsnnet` model**:

- WOW
- S-UNIWARD
- HILL
- HUGO
- MiPOD
- Payloads: 0.2 and 0.4 bpp
- D4 TTA: 8 transforms
- Response signatures: Q10–Q90
- \(\gamma=0.90\)
- Checkpoint loading with `strict=True`
- No resizing after embedding
- Architecture and image-integrity audits before inference

> Run this notebook with the `hsk_torch` kernel.  
> The complete execution may require substantial time because detector responses are recomputed for every image, detector, and payload.


In [ ]:
# ============================================================
# CELL 1 — USER SETTINGS
# ============================================================

from pathlib import Path
import os
import sys

PIPELINE_PATH = (
    r"${HSNNET_LEGACY_ROOT}"
    r"\42_EXBOW2_FULL5_EXTERNAL_READY_AUTOLOCATE"
    r"\scripts\run_candidate_pipeline_full5.py"
)

RESULT_FOLDER = "R2_CANDIDATE_RESULTS_HSNNET_LOCKED_V2"

MAX_IMAGES = 0
BATCH_SIZE = 32
TTA_COUNT = 8
IMAGE_AUDIT_LIMIT = 0

RUN_STAGES = ["all"]

RUN_SMOKE_TEST = False
SMOKE_MAX_IMAGES = 10
SMOKE_AUDIT_LIMIT = 100

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())
print("Pipeline path:", PIPELINE_PATH)
print("Result folder:", RESULT_FOLDER)

In [ ]:
# ============================================================
# 1) USER SETTINGS — EDIT ONLY THIS SECTION
# ============================================================

from pathlib import Path
import os
import sys

# Provide the direct pipeline path when known. Leave empty to enable automatic discovery.
PIPELINE_PATH = r""

# Use a new results directory name to prevent reuse of stale cached outputs.
RESULT_FOLDER = "R2_CANDIDATE_RESULTS_HSNNET_LOCKED_V2"

# Official execution settings
MAX_IMAGES = 0             # 0 = all images
BATCH_SIZE = 32
TTA_COUNT = 8
IMAGE_AUDIT_LIMIT = 0      # 0 = audit all images

# Complete execution:
RUN_STAGES = ["all"]

# To resume an interrupted run at one stage, replace the line above, for example:
# RUN_STAGES = ["boss04"]
# RUN_STAGES = ["boss02"]
# RUN_STAGES = ["bows04"]
# RUN_STAGES = ["bows02"]
# RUN_STAGES = ["table"]

# Quick smoke test; results are not valid for manuscript reporting:
RUN_SMOKE_TEST = False
SMOKE_MAX_IMAGES = 10
SMOKE_AUDIT_LIMIT = 100

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())


## 2) Install the Locked Model Definition and Rebuild Script in the Working Directory

This cell writes two locally locked files:

- `model_definition_hsnnet_accplus_locked.py`
- `rebuild_candidate_bank_hsnnet_locked.py`

Both files are recreated verbatim during each run, preventing the workflow from depending on an outdated model definition elsewhere in the project.


In [ ]:
# ============================================================
# 2) WRITE THE LOCKED FILES
# ============================================================

import base64
from pathlib import Path

MODEL_B64 = """ZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmltcG9ydCBvcwppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgTE9DS0VEIEFDQysgSFNOTkVUIE1PREVMIERFRklOSVRJT04KIyBFeGFjdCBhcmNoaXRlY3R1cmUgdXNlZCBieSB0aGUgYWxnb3JpdGhtLXNwZWNpZmljIGJpbmFyeSBkZXRlY3RvciBiYW5rLgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpQQVlMT0FEID0gIjAuNGJwcCIKUk9PVF9ESVIgPSByIkM6XFVzZXJzXEhTTlxCT1NTYmFzZS0xLjAxIgpDT1ZFUl9ESVIgPSBvcy5wYXRoLmpvaW4oUk9PVF9ESVIsICJjb3ZlciIpClNURUdPX1JPT1QgPSBvcy5wYXRoLmpvaW4oUk9PVF9ESVIsICJzdGVnbyIpCk9VVF9ST09UID0gciIuXG91dHB1dHNfYWNjX3BsdXNfYWxsX2FsZ29yaXRobXNfMDRicHAiCgpBTEdPX0ZPTERFUl9NQVAgPSB7CiAgICAiV09XIjogIldPVyIsCiAgICAiUy1VTklXQVJEIjogIlMtVU5JV0FSRCIsCiAgICAiSFVHTyI6ICJIVUdPIiwKICAgICJNaVBPRCI6ICJNaVBPRCIsCiAgICAiSElMTCI6ICJISUxMIiwKfQoKRVhQRUNURURfVE9UQUxfUEFSQU1FVEVSUyA9IDdfMjkzXzAwOQpFWFBFQ1RFRF9UUkFJTkFCTEVfUEFSQU1FVEVSUyA9IDdfMjkyXzYwOQpFWFBFQ1RFRF9GUk9aRU5fRlJPTlRFTkRfUEFSQU1FVEVSUyA9IDQwMAoKCkBkYXRhY2xhc3MKY2xhc3MgQ0ZHOgogICAgYWxnb3JpdGhtOiBzdHIgPSAiV09XIgogICAgcGF5bG9hZDogc3RyID0gUEFZTE9BRAogICAgcm9vdF9kaXI6IHN0ciA9IFJPT1RfRElSCiAgICBjb3Zlcl9kaXI6IHN0ciA9IENPVkVSX0RJUgogICAgc3RlZ29fcm9vdDogc3RyID0gU1RFR09fUk9PVAogICAgc3RlZ29fZGlyOiBzdHIgPSAiIgogICAgb3V0X3Jvb3Q6IHN0ciA9IE9VVF9ST09UCiAgICBvdXRfZGlyOiBzdHIgPSAiIgoKICAgIGltYWdlX3NpemU6IGludCA9IDI1NgogICAgdHJhaW5fcmF0aW86IGZsb2F0ID0gMC43MAogICAgdmFsX3JhdGlvOiBmbG9hdCA9IDAuMTUKICAgIHRlc3RfcmF0aW86IGZsb2F0ID0gMC4xNQoKICAgIGJhdGNoX3NpemU6IGludCA9IDMyCiAgICBudW1fd29ya2VyczogaW50ID0gMAogICAgcGluX21lbW9yeTogYm9vbCA9IFRydWUKCiAgICBlcG9jaHM6IGludCA9IDE1MAogICAgbHI6IGZsb2F0ID0gOGUtNQogICAgd2VpZ2h0X2RlY2F5OiBmbG9hdCA9IDFlLTQKICAgIHdhcm11cF9lcG9jaHM6IGludCA9IDUKICAgIG1pbl9scl9yYXRpbzogZmxvYXQgPSAwLjAxCgogICAgcGF0aWVuY2U6IGludCA9IDIwCiAgICBlYXJseV9zdG9wX3N0YXJ0X2Vwb2NoOiBpbnQgPSA3MAogICAgZ3JhZF9jbGlwOiBmbG9hdCA9IDEuMAogICAgZW1hX2RlY2F5OiBmbG9hdCA9IDAuOTk5CgogICAgc2VlZDogaW50ID0gNDIKICAgIHVzZV9hbXA6IGJvb2wgPSBUcnVlCiAgICB1c2VfcmVzaXplOiBib29sID0gRmFsc2UKICAgIHNhdmVfcGxvdHM6IGJvb2wgPSBUcnVlCgogICAgdGx1X3RocmVzaG9sZDogZmxvYXQgPSAzLjAKICAgIHNpZ21hX3dpbmRvdzogaW50ID0gMwogICAgbm9ybWFsaXplX2tlcm5lbHM6IGJvb2wgPSBUcnVlCiAgICBmcm9udGVuZF90cmFpbmFibGU6IGJvb2wgPSBGYWxzZQoKICAgIHVzZV9hdWdtZW50YXRpb246IGJvb2wgPSBUcnVlCiAgICBkNF9yZXBlYXRfZmFjdG9yOiBpbnQgPSAyCiAgICBsYWJlbF9zbW9vdGhpbmc6IGZsb2F0ID0gMC4wMgoKICAgIHVzZV9maW5hbF90dGE6IGJvb2wgPSBUcnVlCiAgICB0dGFfZDRfY291bnQ6IGludCA9IDgKCgpkZWYgbWFrZV9jZmdfZm9yX2FsZ29yaXRobShhbGdvOiBzdHIpIC0+IENGRzoKICAgIGlmIGFsZ28gbm90IGluIEFMR09fRk9MREVSX01BUDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVW5rbm93biBhbGdvcml0aG06IHthbGdvfSIpCiAgICBmb2xkZXIgPSBBTEdPX0ZPTERFUl9NQVBbYWxnb10KICAgIHN0ZWdvX2RpciA9IG9zLnBhdGguam9pbihTVEVHT19ST09ULCBmb2xkZXIsIFBBWUxPQUQsICJzdGVnbyIpCiAgICBzYWZlX2FsZ28gPSBhbGdvLnJlcGxhY2UoIi8iLCAiXyIpLnJlcGxhY2UoIlxcIiwgIl8iKQogICAgb3V0X2RpciA9IG9zLnBhdGguam9pbihPVVRfUk9PVCwgZiJvdXRwdXRzX3RlbXBsYXRlX3tzYWZlX2FsZ299X2FjY19wbHVzIikKICAgIHJldHVybiBDRkcoCiAgICAgICAgYWxnb3JpdGhtPWFsZ28sCiAgICAgICAgcGF5bG9hZD1QQVlMT0FELAogICAgICAgIHN0ZWdvX2Rpcj1zdGVnb19kaXIsCiAgICAgICAgb3V0X2Rpcj1vdXRfZGlyLAogICAgKQoKCmRlZiBfbm9ybWFsaXplX2tlcm5lbF9sMShrOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgIHMgPSBrLmFicygpLnN1bSgpCiAgICByZXR1cm4gayAvIHMgaWYgcyA+IDAgZWxzZSBrCgoKZGVmIF9wbGFjZV9saW5lX2tlcm5lbCh2YWx1ZXMsIGRpcmVjdGlvbj0iaCIsIG1vZGU9InNob3J0IikgLT4gdG9yY2guVGVuc29yOgogICAgayA9IHRvcmNoLnplcm9zKCg1LCA1KSwgZHR5cGU9dG9yY2guZmxvYXQzMikKICAgIHZhbHVlcyA9IHRvcmNoLnRlbnNvcih2YWx1ZXMsIGR0eXBlPXRvcmNoLmZsb2F0MzIpCgogICAgaWYgbW9kZSA9PSAic2hvcnQiOgogICAgICAgIGlkeCA9IFsxLCAyLCAzXQogICAgZWxpZiBtb2RlID09ICJsb25nIjoKICAgICAgICBpZHggPSBbMCwgMiwgNF0KICAgIGVsaWYgbW9kZSA9PSAiZnVsbCI6CiAgICAgICAgaWR4ID0gWzAsIDEsIDIsIDMsIDRdCiAgICBlbHNlOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1vZGUgbXVzdCBiZSBzaG9ydCwgbG9uZywgb3IgZnVsbCIpCgogICAgaWYgbGVuKHZhbHVlcykgIT0gbGVuKGlkeCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiS2VybmVsIGNvZWZmaWNpZW50IGNvdW50IGRvZXMgbm90IG1hdGNoIHBsYWNlbWVudCBtb2RlIikKCiAgICBpZiBkaXJlY3Rpb24gPT0gImgiOgogICAgICAgIGZvciBjLCB2IGluIHppcChpZHgsIHZhbHVlcyk6CiAgICAgICAgICAgIGtbMiwgY10gPSB2CiAgICBlbGlmIGRpcmVjdGlvbiA9PSAidiI6CiAgICAgICAgZm9yIHIsIHYgaW4gemlwKGlkeCwgdmFsdWVzKToKICAgICAgICAgICAga1tyLCAyXSA9IHYKICAgIGVsaWYgZGlyZWN0aW9uID09ICJkIjoKICAgICAgICBmb3IgaSwgdiBpbiB6aXAoaWR4LCB2YWx1ZXMpOgogICAgICAgICAgICBrW2ksIGldID0gdgogICAgZWxpZiBkaXJlY3Rpb24gPT0gImFkIjoKICAgICAgICBmb3IgaSwgdiBpbiB6aXAoaWR4LCB2YWx1ZXMpOgogICAgICAgICAgICBrW2ksIDQgLSBpXSA9IHYKICAgIGVsc2U6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZGlyZWN0aW9uIG11c3QgYmUgaCwgdiwgZCwgb3IgYWQiKQogICAgcmV0dXJuIGsKCgpkZWYgZ2V0X3N0YXRpc3RpY2FsX3ByZWRpY3Rpb25fZmlsdGVycyhub3JtYWxpemU6IGJvb2wgPSBUcnVlKSAtPiB0b3JjaC5UZW5zb3I6CiAgICBrZXJuZWxzID0gW10KCiAgICBkZWYgYWRkKGspOgogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKGssIHRvcmNoLlRlbnNvcik6CiAgICAgICAgICAgIGsgPSB0b3JjaC50ZW5zb3IoaywgZHR5cGU9dG9yY2guZmxvYXQzMikKICAgICAgICBlbHNlOgogICAgICAgICAgICBrID0gay5jbG9uZSgpLmZsb2F0KCkKICAgICAgICBrZXJuZWxzLmFwcGVuZChfbm9ybWFsaXplX2tlcm5lbF9sMShrKSBpZiBub3JtYWxpemUgZWxzZSBrKQoKICAgIGFkZChfcGxhY2VfbGluZV9rZXJuZWwoWy0wLjUsIDEuMCwgLTAuNV0sICJoIiwgInNob3J0IikpCiAgICBhZGQoX3BsYWNlX2xpbmVfa2VybmVsKFstMC41LCAxLjAsIC0wLjVdLCAidiIsICJzaG9ydCIpKQogICAgYWRkKF9wbGFjZV9saW5lX2tlcm5lbChbLTAuNSwgMS4wLCAtMC41XSwgImQiLCAic2hvcnQiKSkKICAgIGFkZChfcGxhY2VfbGluZV9rZXJuZWwoWy0wLjUsIDEuMCwgLTAuNV0sICJhZCIsICJzaG9ydCIpKQoKICAgIGFkZChbCiAgICAgICAgWzAsIDAsIDAsIDAsIDBdLAogICAgICAgIFswLCAwLCAtMC4yNSwgMCwgMF0sCiAgICAgICAgWzAsIC0wLjI1LCAxLjAsIC0wLjI1LCAwXSwKICAgICAgICBbMCwgMCwgLTAuMjUsIDAsIDBdLAogICAgICAgIFswLCAwLCAwLCAwLCAwXSwKICAgIF0pCiAgICBhZGQoWwogICAgICAgIFswLCAwLCAwLCAwLCAwXSwKICAgICAgICBbMCwgLTAuMTI1LCAtMC4xMjUsIC0wLjEyNSwgMF0sCiAgICAgICAgWzAsIC0wLjEyNSwgMS4wLCAtMC4xMjUsIDBdLAogICAgICAgIFswLCAtMC4xMjUsIC0wLjEyNSwgLTAuMTI1LCAwXSwKICAgICAgICBbMCwgMCwgMCwgMCwgMF0sCiAgICBdKQoKICAgIGFkZChfcGxhY2VfbGluZV9rZXJuZWwoWy0wLjUsIDEuMCwgLTAuNV0sICJoIiwgImxvbmciKSkKICAgIGFkZChfcGxhY2VfbGluZV9rZXJuZWwoWy0wLjUsIDEuMCwgLTAuNV0sICJ2IiwgImxvbmciKSkKICAgIGFkZChfcGxhY2VfbGluZV9rZXJuZWwoWy0wLjUsIDEuMCwgLTAuNV0sICJkIiwgImxvbmciKSkKICAgIGFkZChfcGxhY2VfbGluZV9rZXJuZWwoWy0wLjUsIDEuMCwgLTAuNV0sICJhZCIsICJsb25nIikpCgogICAgYWRkKF9wbGFjZV9saW5lX2tlcm5lbCgKICAgICAgICBbLTAuMjUsIC0wLjI1LCAxLjAsIC0wLjI1LCAtMC4yNV0sICJoIiwgImZ1bGwiCiAgICApKQogICAgYWRkKF9wbGFjZV9saW5lX2tlcm5lbCgKICAgICAgICBbLTAuMjUsIC0wLjI1LCAxLjAsIC0wLjI1LCAtMC4yNV0sICJ2IiwgImZ1bGwiCiAgICApKQoKICAgIGFkZChfcGxhY2VfbGluZV9rZXJuZWwoCiAgICAgICAgWzAuMjUsIC0wLjc1LCAxLjAsIC0wLjc1LCAwLjI1XSwgImgiLCAiZnVsbCIKICAgICkpCiAgICBhZGQoX3BsYWNlX2xpbmVfa2VybmVsKAogICAgICAgIFswLjI1LCAtMC43NSwgMS4wLCAtMC43NSwgMC4yNV0sICJ2IiwgImZ1bGwiCiAgICApKQogICAgYWRkKF9wbGFjZV9saW5lX2tlcm5lbCgKICAgICAgICBbMC4yNSwgLTAuNzUsIDEuMCwgLTAuNzUsIDAuMjVdLCAiZCIsICJmdWxsIgogICAgKSkKICAgIGFkZChfcGxhY2VfbGluZV9rZXJuZWwoCiAgICAgICAgWzAuMjUsIC0wLjc1LCAxLjAsIC0wLjc1LCAwLjI1XSwgImFkIiwgImZ1bGwiCiAgICApKQoKICAgIHJldHVybiB0b3JjaC5zdGFjayhrZXJuZWxzLCBkaW09MCkudW5zcXVlZXplKDEpCgoKY2xhc3MgU3RhdGlzdGljYWxSZXNpZHVhbEZyb250RW5kKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBpbl9jaGFubmVscz0xLAogICAgICAgIHRsdV90aHJlc2hvbGQ9My4wLAogICAgICAgIHNpZ21hX3dpbmRvdz0zLAogICAgICAgIGVwcz0xZS00LAogICAgICAgIG5vcm1hbGl6ZV9rZXJuZWxzPVRydWUsCiAgICAgICAgdHJhaW5hYmxlPUZhbHNlLAogICAgKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBpZiBzaWdtYV93aW5kb3cgJSAyID09IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNpZ21hX3dpbmRvdyBtdXN0IGJlIG9kZCIpCgogICAgICAgIGtlcm5lbHMgPSBnZXRfc3RhdGlzdGljYWxfcHJlZGljdGlvbl9maWx0ZXJzKAogICAgICAgICAgICBub3JtYWxpemU9bm9ybWFsaXplX2tlcm5lbHMKICAgICAgICApCiAgICAgICAgc2VsZi5jb252ID0gbm4uQ29udjJkKAogICAgICAgICAgICBpbl9jaGFubmVscywKICAgICAgICAgICAga2VybmVscy5zaGFwZVswXSwKICAgICAgICAgICAga2VybmVsX3NpemU9NSwKICAgICAgICAgICAgc3RyaWRlPTEsCiAgICAgICAgICAgIHBhZGRpbmc9MiwKICAgICAgICAgICAgYmlhcz1GYWxzZSwKICAgICAgICApCgogICAgICAgIGlmIGluX2NoYW5uZWxzICE9IDE6CiAgICAgICAgICAgIGtlcm5lbHMgPSBrZXJuZWxzLnJlcGVhdCgxLCBpbl9jaGFubmVscywgMSwgMSkgLyBmbG9hdChpbl9jaGFubmVscykKCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIHNlbGYuY29udi53ZWlnaHQuY29weV8oa2VybmVscykKCiAgICAgICAgc2VsZi5jb252LndlaWdodC5yZXF1aXJlc19ncmFkID0gYm9vbCh0cmFpbmFibGUpCiAgICAgICAgc2VsZi50bHVfdGhyZXNob2xkID0gZmxvYXQodGx1X3RocmVzaG9sZCkKICAgICAgICBzZWxmLnNpZ21hX3dpbmRvdyA9IGludChzaWdtYV93aW5kb3cpCiAgICAgICAgc2VsZi5lcHMgPSBmbG9hdChlcHMpCgogICAgZGVmIGxvY2FsX3ZhcmlhbmNlX25vcm1hbGl6ZShzZWxmLCByZXNpZHVhbCk6CiAgICAgICAgcGFkID0gc2VsZi5zaWdtYV93aW5kb3cgLy8gMgogICAgICAgIG11ID0gRi5hdmdfcG9vbDJkKAogICAgICAgICAgICByZXNpZHVhbCwKICAgICAgICAgICAgc2VsZi5zaWdtYV93aW5kb3csCiAgICAgICAgICAgIHN0cmlkZT0xLAogICAgICAgICAgICBwYWRkaW5nPXBhZCwKICAgICAgICApCiAgICAgICAgbXUyID0gRi5hdmdfcG9vbDJkKAogICAgICAgICAgICByZXNpZHVhbCAqIHJlc2lkdWFsLAogICAgICAgICAgICBzZWxmLnNpZ21hX3dpbmRvdywKICAgICAgICAgICAgc3RyaWRlPTEsCiAgICAgICAgICAgIHBhZGRpbmc9cGFkLAogICAgICAgICkKICAgICAgICB2YXJpYW5jZSA9IHRvcmNoLmNsYW1wKG11MiAtIG11ICogbXUsIG1pbj0wLjApCiAgICAgICAgc3RkID0gdG9yY2guc3FydCh2YXJpYW5jZSArIHNlbGYuZXBzKQogICAgICAgIHJldHVybiByZXNpZHVhbCAvIChzdGQgKyBzZWxmLmVwcykKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICByZXNpZHVhbCA9IHNlbGYuY29udih4KQogICAgICAgIG5vcm1hbGl6ZWQgPSBzZWxmLmxvY2FsX3ZhcmlhbmNlX25vcm1hbGl6ZShyZXNpZHVhbCkKICAgICAgICByZXR1cm4gdG9yY2guY2xhbXAoCiAgICAgICAgICAgIG5vcm1hbGl6ZWQsCiAgICAgICAgICAgIC1zZWxmLnRsdV90aHJlc2hvbGQsCiAgICAgICAgICAgIHNlbGYudGx1X3RocmVzaG9sZCwKICAgICAgICApCgoKY2xhc3MgQ29udkJOUmVMVShubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2NoLCBvdXRfY2gsIGs9Mywgcz0xLCBwPTEpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYmxvY2sgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoCiAgICAgICAgICAgICAgICBpbl9jaCwKICAgICAgICAgICAgICAgIG91dF9jaCwKICAgICAgICAgICAgICAgIGssCiAgICAgICAgICAgICAgICBzdHJpZGU9cywKICAgICAgICAgICAgICAgIHBhZGRpbmc9cCwKICAgICAgICAgICAgICAgIGJpYXM9RmFsc2UsCiAgICAgICAgICAgICksCiAgICAgICAgICAgIG5uLkJhdGNoTm9ybTJkKG91dF9jaCksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYuYmxvY2soeCkKCgpjbGFzcyBTRUJsb2NrKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgY2g6IGludCwgcmVkdWN0aW9uOiBpbnQgPSAxNik6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgaGlkZGVuID0gbWF4KGNoIC8vIHJlZHVjdGlvbiwgOCkKICAgICAgICBzZWxmLnBvb2wgPSBubi5BZGFwdGl2ZUF2Z1Bvb2wyZCgxKQogICAgICAgIHNlbGYuZmMgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoY2gsIGhpZGRlbiwgMSwgYmlhcz1UcnVlKSwKICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICAgICBubi5Db252MmQoaGlkZGVuLCBjaCwgMSwgYmlhcz1UcnVlKSwKICAgICAgICAgICAgbm4uU2lnbW9pZCgpLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICByZXR1cm4geCAqIHNlbGYuZmMoc2VsZi5wb29sKHgpKQoKCmNsYXNzIFJlc2lkdWFsQmxvY2sobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjaCwgdXNlX3NlPVRydWUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuY29udjEgPSBubi5Db252MmQoY2gsIGNoLCAzLCBwYWRkaW5nPTEsIGJpYXM9RmFsc2UpCiAgICAgICAgc2VsZi5ibjEgPSBubi5CYXRjaE5vcm0yZChjaCkKICAgICAgICBzZWxmLmNvbnYyID0gbm4uQ29udjJkKGNoLCBjaCwgMywgcGFkZGluZz0xLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYuYm4yID0gbm4uQmF0Y2hOb3JtMmQoY2gpCiAgICAgICAgc2VsZi5zZSA9IFNFQmxvY2soY2gpIGlmIHVzZV9zZSBlbHNlIG5uLklkZW50aXR5KCkKICAgICAgICBzZWxmLnJlbHUgPSBubi5SZUxVKGlucGxhY2U9VHJ1ZSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICBvdXQgPSBzZWxmLnJlbHUoc2VsZi5ibjEoc2VsZi5jb252MSh4KSkpCiAgICAgICAgb3V0ID0gc2VsZi5ibjIoc2VsZi5jb252MihvdXQpKQogICAgICAgIG91dCA9IHNlbGYuc2Uob3V0KSArIHgKICAgICAgICByZXR1cm4gc2VsZi5yZWx1KG91dCkKCgpjbGFzcyBBZGFwdGl2ZUNvbmNhdFBvb2wyZChubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYXZnID0gbm4uQWRhcHRpdmVBdmdQb29sMmQoKDEsIDEpKQogICAgICAgIHNlbGYubWF4ID0gbm4uQWRhcHRpdmVNYXhQb29sMmQoKDEsIDEpKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHJldHVybiB0b3JjaC5jYXQoW3NlbGYuYXZnKHgpLCBzZWxmLm1heCh4KV0sIGRpbT0xKQoKCmNsYXNzIGhzbm5ldChubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNmZzogQ0ZHKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmZyb250ZW5kID0gU3RhdGlzdGljYWxSZXNpZHVhbEZyb250RW5kKAogICAgICAgICAgICBpbl9jaGFubmVscz0xLAogICAgICAgICAgICB0bHVfdGhyZXNob2xkPWNmZy50bHVfdGhyZXNob2xkLAogICAgICAgICAgICBzaWdtYV93aW5kb3c9Y2ZnLnNpZ21hX3dpbmRvdywKICAgICAgICAgICAgZXBzPTFlLTQsCiAgICAgICAgICAgIG5vcm1hbGl6ZV9rZXJuZWxzPWNmZy5ub3JtYWxpemVfa2VybmVscywKICAgICAgICAgICAgdHJhaW5hYmxlPWNmZy5mcm9udGVuZF90cmFpbmFibGUsCiAgICAgICAgKQoKICAgICAgICBzZWxmLnN0ZW0gPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBDb252Qk5SZUxVKDE2LCAzMiksCiAgICAgICAgICAgIFJlc2lkdWFsQmxvY2soMzIpLAogICAgICAgICAgICBubi5BdmdQb29sMmQoMiwgMiksCiAgICAgICAgKQogICAgICAgIHNlbGYuc3RhZ2UyID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgQ29udkJOUmVMVSgzMiwgNjQpLAogICAgICAgICAgICBSZXNpZHVhbEJsb2NrKDY0KSwKICAgICAgICAgICAgUmVzaWR1YWxCbG9jayg2NCksCiAgICAgICAgICAgIG5uLkF2Z1Bvb2wyZCgyLCAyKSwKICAgICAgICApCiAgICAgICAgc2VsZi5zdGFnZTMgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBDb252Qk5SZUxVKDY0LCAxMjgpLAogICAgICAgICAgICBSZXNpZHVhbEJsb2NrKDEyOCksCiAgICAgICAgICAgIFJlc2lkdWFsQmxvY2soMTI4KSwKICAgICAgICAgICAgbm4uQXZnUG9vbDJkKDIsIDIpLAogICAgICAgICkKICAgICAgICBzZWxmLnN0YWdlNCA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIENvbnZCTlJlTFUoMTI4LCAyNTYpLAogICAgICAgICAgICBSZXNpZHVhbEJsb2NrKDI1NiksCiAgICAgICAgICAgIFJlc2lkdWFsQmxvY2soMjU2KSwKICAgICAgICAgICAgbm4uQXZnUG9vbDJkKDIsIDIpLAogICAgICAgICkKICAgICAgICBzZWxmLnN0YWdlNSA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIENvbnZCTlJlTFUoMjU2LCAzODQpLAogICAgICAgICAgICBSZXNpZHVhbEJsb2NrKDM4NCksCiAgICAgICAgKQoKICAgICAgICBzZWxmLnBvb2wgPSBBZGFwdGl2ZUNvbmNhdFBvb2wyZCgpCiAgICAgICAgc2VsZi5oZWFkID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uRmxhdHRlbigpLAogICAgICAgICAgICBubi5Ecm9wb3V0KDAuMjUpLAogICAgICAgICAgICBubi5MaW5lYXIoMzg0ICogMiwgMjU2KSwKICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICAgICBubi5Ecm9wb3V0KDAuMjApLAogICAgICAgICAgICBubi5MaW5lYXIoMjU2LCAxKSwKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgeCA9IHNlbGYuZnJvbnRlbmQoeCkKICAgICAgICB4ID0gc2VsZi5zdGVtKHgpCiAgICAgICAgeCA9IHNlbGYuc3RhZ2UyKHgpCiAgICAgICAgeCA9IHNlbGYuc3RhZ2UzKHgpCiAgICAgICAgeCA9IHNlbGYuc3RhZ2U0KHgpCiAgICAgICAgeCA9IHNlbGYuc3RhZ2U1KHgpCiAgICAgICAgeCA9IHNlbGYucG9vbCh4KQogICAgICAgIHJldHVybiBzZWxmLmhlYWQoeCkKCgpIYXNzYW5OZXRWMl9TUk0gPSBoc25uZXQKCgpkZWYgX2FwcGx5X2Q0X2JhdGNoKHgsIGc6IGludCk6CiAgICBpZiBnID09IDA6CiAgICAgICAgcmV0dXJuIHgKICAgIGlmIGcgPT0gMToKICAgICAgICByZXR1cm4gdG9yY2gucm90OTAoeCwgaz0xLCBkaW1zPVsyLCAzXSkKICAgIGlmIGcgPT0gMjoKICAgICAgICByZXR1cm4gdG9yY2gucm90OTAoeCwgaz0yLCBkaW1zPVsyLCAzXSkKICAgIGlmIGcgPT0gMzoKICAgICAgICByZXR1cm4gdG9yY2gucm90OTAoeCwgaz0zLCBkaW1zPVsyLCAzXSkKICAgIGlmIGcgPT0gNDoKICAgICAgICByZXR1cm4gdG9yY2guZmxpcCh4LCBkaW1zPVszXSkKICAgIGlmIGcgPT0gNToKICAgICAgICByZXR1cm4gdG9yY2guZmxpcCh4LCBkaW1zPVsyXSkKICAgIGlmIGcgPT0gNjoKICAgICAgICByZXR1cm4gdG9yY2gucm90OTAoCiAgICAgICAgICAgIHRvcmNoLmZsaXAoeCwgZGltcz1bM10pLAogICAgICAgICAgICBrPTEsCiAgICAgICAgICAgIGRpbXM9WzIsIDNdLAogICAgICAgICkKICAgIGlmIGcgPT0gNzoKICAgICAgICByZXR1cm4gdG9yY2gucm90OTAoCiAgICAgICAgICAgIHRvcmNoLmZsaXAoeCwgZGltcz1bMl0pLAogICAgICAgICAgICBrPTEsCiAgICAgICAgICAgIGRpbXM9WzIsIDNdLAogICAgICAgICkKICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJENCBlbGVtZW50IGluZGV4IG91dCBvZiByYW5nZToge2d9IikKCgpkZWYgYXJjaGl0ZWN0dXJlX2NvdW50cyhjZmc6IENGRyB8IE5vbmUgPSBOb25lKToKICAgIGNmZyA9IGNmZyBvciBDRkcoKQogICAgbW9kZWwgPSBoc25uZXQoY2ZnKQogICAgdG90YWwgPSBzdW0ocC5udW1lbCgpIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSkKICAgIHRyYWluYWJsZSA9IHN1bSgKICAgICAgICBwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpIGlmIHAucmVxdWlyZXNfZ3JhZAogICAgKQogICAgZnJvbnRlbmQgPSBzdW0ocC5udW1lbCgpIGZvciBwIGluIG1vZGVsLmZyb250ZW5kLnBhcmFtZXRlcnMoKSkKICAgIHJldHVybiB7CiAgICAgICAgInRvdGFsX3BhcmFtZXRlcnMiOiBpbnQodG90YWwpLAogICAgICAgICJ0cmFpbmFibGVfcGFyYW1ldGVycyI6IGludCh0cmFpbmFibGUpLAogICAgICAgICJmcm9udGVuZF9wYXJhbWV0ZXJzIjogaW50KGZyb250ZW5kKSwKICAgIH0K"""
RUNNER_B64 = """..."""

WORK_DIR = Path.cwd() / "HSNNET_LOCKED_BANK_NOTEBOOK"
WORK_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DEFINITION_PATH = WORK_DIR / "src/model_definition_hsnnet_locked.py"
RUNNER_PATH = WORK_DIR / "scripts/rebuild_candidate_bank.py"

MODEL_DEFINITION_PATH.write_bytes(base64.b64decode(MODEL_B64))
RUNNER_PATH.write_bytes(base64.b64decode(RUNNER_B64))

print("Locked model :", MODEL_DEFINITION_PATH)
print("Locked runner:", RUNNER_PATH)
print("Files written successfully.")


## 3) Locate the Pipeline and Perform the Preflight Audit

This cell:

1. Locates `run_candidate_pipeline_full5.py`.
2. Verifies CUDA and PyTorch availability.
3. Imports the locked `hsnnet` definition.
4. Verifies the official parameter counts.
5. Displays the BOSSBase, BOWS2, and checkpoint paths registered in the pipeline before execution.


In [ ]:
# ============================================================
# 3) PREFLIGHT
# ============================================================

import importlib.util
import sys
from pathlib import Path


def import_from_path(path: Path, module_name: str):
    spec = importlib.util.spec_from_file_location(module_name, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot import: {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def locate_pipeline(explicit_path: str = "") -> Path:
    if explicit_path.strip():
        p = Path(explicit_path)
        if not p.is_file():
            raise FileNotFoundError(f"PIPELINE_PATH does not exist: {p}")
        return p.resolve()

    candidates = [
        Path.cwd() / "run_candidate_pipeline_full5.py",
        Path.cwd() / "scripts" / "run_candidate_pipeline_full5.py",
        Path(r"${HSNNET_LEGACY_ROOT}\42_EXBOW2_FULL5_EXTERNAL_READY\scripts\run_candidate_pipeline_full5.py"),
        Path(r"scripts/run_candidate_pipeline_full5.py"),
        Path(r"${HSNNET_LEGACY_ROOT}\42_EXBOW2_CANDIDATE_READY\run_candidate_pipeline_full5.py"),
        Path(r"${HSNNET_LEGACY_ROOT}\42_EXBOW2_CANDIDATE_READY\scripts\run_candidate_pipeline_full5.py"),
    ]

    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    root = Path(r"${HSNNET_LEGACY_ROOT}")
    if root.is_dir():
        matches = sorted(root.rglob("run_candidate_pipeline_full5.py"))
        if matches:
            print("Possible pipeline files:")
            for i, item in enumerate(matches, start=1):
                print(f"[{i}] {item}")
            print("Using the first match. Set PIPELINE_PATH explicitly if this is not the locked final file.")
            return matches[0].resolve()

    raise FileNotFoundError(
        "run_candidate_pipeline_full5.py was not found. "
        "Set PIPELINE_PATH in Cell 1."
    )


PIPELINE_FILE = locate_pipeline(PIPELINE_PATH)
print("=" * 110)
print("Candidate pipeline:", PIPELINE_FILE)
print("=" * 110)

# Required packages
import torch
import numpy as np
from PIL import Image
import pandas as pd

print("Torch version :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device   :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "CUDA is not available in this Jupyter kernel. "
        "Select the hsk_torch kernel before the full run."
    )

# Verify the locked architecture
model_module = import_from_path(
    MODEL_DEFINITION_PATH,
    "hsnnet_accplus_notebook_preflight",
)
counts = model_module.architecture_counts()
print("\nLocked architecture counts:")
print(counts)

assert counts["total_parameters"] == 7_293_009
assert counts["trainable_parameters"] == 7_292_609
assert counts["frontend_parameters"] == 400

# Verify the expected pipeline functions before running it.
pipeline_text = PIPELINE_FILE.read_text(encoding="utf-8", errors="replace")
required_tokens = [
    "run_internal",
    "run_external",
    "build_manifest",
    "make_reviewer_table",
    "clean_state_dict",
    "build_cfg_obj",
]
missing_tokens = [token for token in required_tokens if token not in pipeline_text]
if missing_tokens:
    raise RuntimeError(
        "This does not appear to be the required full-five candidate pipeline. "
        f"Missing tokens: {missing_tokens}"
    )

pipeline_module = import_from_path(
    PIPELINE_FILE,
    "candidate_pipeline_notebook_preflight",
)

PIPELINE_BASE = Path(pipeline_module.BASE)
RESULT_ROOT = PIPELINE_BASE / RESULT_FOLDER

print("\nPipeline BASE :", PIPELINE_BASE)
print("Result root   :", RESULT_ROOT)

for attr in ["BOSS_COVER", "BOWS2_COVER", "REFERENCE_MODEL"]:
    if hasattr(pipeline_module, attr):
        print(f"{attr:16s}:", getattr(pipeline_module, attr))

if hasattr(pipeline_module, "BOSS_STEGO_DIRS"):
    print("\nBOSSBase stego roots are registered for:", list(pipeline_module.BOSS_STEGO_DIRS))
if hasattr(pipeline_module, "BOWS2_STEGO_DIRS"):
    print("BOWS2 stego roots are registered for  :", list(pipeline_module.BOWS2_STEGO_DIRS))

print("\nPREFLIGHT STATUS: PASS")


## 4) Jupyter Execution Function

This function streams the pipeline output directly into the notebook and saves a text log for each execution stage.


In [ ]:
# ============================================================
# 4) STREAMED SUBPROCESS RUNNER
# ============================================================

import subprocess
import sys
import time
from pathlib import Path


def run_locked_stage(
    stage: str,
    *,
    result_folder: str = RESULT_FOLDER,
    max_images: int = MAX_IMAGES,
    batch_size: int = BATCH_SIZE,
    tta_count: int = TTA_COUNT,
    image_audit_limit: int = IMAGE_AUDIT_LIMIT,
):
    command = [
        sys.executable,
        str(RUNNER_PATH),
        "--stage", stage,
        "--pipeline", str(PIPELINE_FILE),
        "--model-definition", str(MODEL_DEFINITION_PATH),
        "--result-folder", result_folder,
        "--max-images", str(max_images),
        "--batch-size", str(batch_size),
        "--tta-count", str(tta_count),
        "--image-audit-limit", str(image_audit_limit),
    ]

    log_dir = WORK_DIR / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{stage}_{time.strftime('%Y%m%d_%H%M%S')}.log"

    print("=" * 110)
    print("RUNNING STAGE:", stage)
    print("COMMAND:")
    print(" ".join(f'"{item}"' if " " in item else item for item in command))
    print("LOG:", log_path)
    print("=" * 110)

    started = time.time()
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )

        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
            log_file.flush()

        return_code = process.wait()

    elapsed = time.time() - started
    print("\n" + "=" * 110)
    print(f"STAGE {stage} FINISHED | return_code={return_code} | elapsed={elapsed/3600:.2f} hours")
    print("=" * 110)

    if return_code != 0:
        raise RuntimeError(
            f"Stage {stage} failed with return code {return_code}. "
            f"Review: {log_path}"
        )

    return log_path


## 5) Optional Smoke Test

Smoke-test results must not be used in the manuscript. This test only verifies that the configured paths, checkpoints, and CUDA environment operate correctly.


In [ ]:
# ============================================================
# 5) OPTIONAL SMOKE TEST
# ============================================================

if RUN_SMOKE_TEST:
    smoke_folder = RESULT_FOLDER + "_SMOKE"
    run_locked_stage(
        "boss04",
        result_folder=smoke_folder,
        max_images=SMOKE_MAX_IMAGES,
        batch_size=BATCH_SIZE,
        tta_count=8,
        image_audit_limit=SMOKE_AUDIT_LIMIT,
    )
else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST=True in Cell 1 to enable it.")


## 6) Complete Official Execution

The default setting `RUN_STAGES = ["all"]` executes the following stages sequentially:

1. Architecture-integrity audit.
2. Image-integrity audit.
3. BOSSBase 0.2 bpp.
4. BOSSBase 0.4 bpp.
5. BOWS2 0.4 bpp.
6. BOWS2 0.2 bpp.
7. Reviewer table generation.

After an interrupted run, select an individual stage in the settings cell and execute this cell again.


In [ ]:
# ============================================================
# 6) FULL OFFICIAL RUN
# ============================================================

completed_logs = []

for stage in RUN_STAGES:
    completed_logs.append(
        run_locked_stage(
            stage,
            result_folder=RESULT_FOLDER,
            max_images=MAX_IMAGES,
            batch_size=BATCH_SIZE,
            tta_count=TTA_COUNT,
            image_audit_limit=IMAGE_AUDIT_LIMIT,
        )
    )

print("\nCompleted logs:")
for item in completed_logs:
    print(" -", item)


## 7) Display Audit and Result Files

This cell displays:

- The architecture-integrity report.
- The image-integrity report.
- The run manifest.
- Available summary, per-algorithm, and reviewer-table files.


In [ ]:
# ============================================================
# 7) INSPECT OUTPUTS
# ============================================================

from pathlib import Path
import json
import pandas as pd
from IPython.display import display

RESULT_ROOT = PIPELINE_BASE / RESULT_FOLDER
print("Result root:", RESULT_ROOT)

if not RESULT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Result directory does not exist yet: {RESULT_ROOT}"
    )

all_files = sorted(path for path in RESULT_ROOT.rglob("*") if path.is_file())

print(f"\nGenerated files: {len(all_files)}")
for path in all_files:
    print(path.relative_to(RESULT_ROOT))

print("\n" + "=" * 110)
print("AUDIT TABLES")
print("=" * 110)

audit_csvs = [
    RESULT_ROOT / "00_LOCKED_INTEGRITY_AUDIT" / "architecture_integrity_audit.csv",
    RESULT_ROOT / "00_LOCKED_INTEGRITY_AUDIT" / "image_integrity_audit.csv",
]

for csv_path in audit_csvs:
    if csv_path.is_file():
        print("\n", csv_path.name)
        display(pd.read_csv(csv_path))

manifest_path = RESULT_ROOT / "locked_run_manifest.json"
if manifest_path.is_file():
    print("\nLOCKED RUN MANIFEST")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(json.dumps(manifest, indent=2, ensure_ascii=False))

# Display likely final result tables.
patterns = [
    "*reviewer*.csv",
    "*final_summary*.csv",
    "*final_per_algorithm*.csv",
    "*ranked_candidate_analysis_per_algorithm*.csv",
    "*summary*.csv",
]

displayed = set()
for pattern in patterns:
    for csv_path in sorted(RESULT_ROOT.rglob(pattern)):
        key = str(csv_path.resolve()).lower()
        if key in displayed:
            continue
        displayed.add(key)
        try:
            frame = pd.read_csv(csv_path)
        except Exception as exc:
            print("Could not read:", csv_path, exc)
            continue

        print("\n", csv_path.relative_to(RESULT_ROOT))
        if len(frame) <= 500:
            display(frame)
        else:
            print(f"Rows: {len(frame)} — displaying first 50")
            display(frame.head(50))


## Criteria for Accepting Results in the Manuscript

Results are accepted only when all of the following checks report **PASS**:

- `strict_load_pass = True`
- `parameter_count_pass = True`
- `missing_keys = 0`
- `unexpected_keys = 0`
- `shape_mismatches = 0`
- All images are grayscale and \(256\times256\)
- `strict_no_resize = True`
- `tta_count = 8`
- `q_low = 10`, `q_high = 90`
- `gamma = 0.90`

After execution, compare the regenerated results against the archived manuscript tables before changing the manuscript. Archived values must never be replaced automatically without verification.
